# Example: Exploratory analysis after embeddings

**Before you start:** Run from `Notebooks/examples/`. If you are new to the pipeline, skim [02_phenotyping_api_usage.ipynb](02_phenotyping_api_usage.ipynb) first.

**In plain terms:** The vision model turns each image into an **embedding** (a numeric summary of appearance). This notebook shows typical **next steps**: summarize hand-crafted **properties** by treatment, optional **clustering** in embedding space, **prototype** images that represent each group, **distance to a reference** (e.g. control) plus **outlier** checks, and two styles of **explainability** that link embeddings to measured biology.

For one **HTML report** that bundles many sections, use `generate_report()` and the [Report Guide](../../docs/guides/report-guide.md).

**Author:** Aitor González-Marfil ([@AAitorG](https://github.com/AAitorG))

---

## 1. Setup

In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from phenome import PhenoMe, get_metadata_from_path, load_dinov2_model

## 2. Load data, embed, optional properties

Same pattern as the other examples: discover files, run `process_images`, then optionally `compute_properties` if you linked **masks** in `find_files` (required for presets like `basic`). Adjust `reference_filters` so the **control** label matches your metadata (here: `condition == "Control"`).

In [ ]:
model, wrapper = load_dinov2_model(model_name="dinov2_vitb14_reg")
pheno = PhenoMe(seed=42)

metadata_fn = get_metadata_from_path(".../(condition)/(filename).*")
image_dir = "path/to/images"
mask_dir = "path/to/masks"

file_df = pheno.find_files(
    image_dir,
    metadata_fn=metadata_fn,
    mask_dir=mask_dir if mask_dir and os.path.exists(mask_dir) else None,
)
print(f"Found {len(file_df)} images")

pheno.process_images(wrapper, channel_mode="split")

has_masks = "mask_path" in file_df.columns and file_df["mask_path"].notna().any()
if has_masks:
    pheno.compute_properties(property_preset="basic")
else:
    print("Skipping compute_properties: add mask_dir in find_files for mask-based presets.")

# Use the same keys in reference_filters, grouped summaries, and plots below.
reference_filters = {"condition": "Control"}

## 3. Summaries by experimental group (needs properties)

These calls answer: *how do measured morphology features differ between conditions?* They require `compute_properties` above.

In [ ]:
prop_keys = pheno.get_available_property_keys()
if not prop_keys:
    print("Skip: run compute_properties (with masks) to enable grouped tables and plots.")
else:
    df_grouped = pheno.filter_properties_by_group(group_by=["condition"])
    pheno.print_property_stats_by_group(
        df_grouped,
        properties=prop_keys[: min(8, len(prop_keys))],
    )
    pheno.plot_counts(group_by=["condition"])
    pheno.top_properties_different_from_reference(
        df_grouped,
        reference_group=reference_filters,
        k=5,
    )

## 4. Cluster in embedding space

Clusters are **data-driven** groups in embedding space; they may or may not match your experimental labels. Set `n_clusters` between 2 and the number of images (see [Pipeline API](../../docs/reference/api/pipeline.md)). After clustering, labels are stored under metadata key `cluster`.

In [ ]:
n_samples = len(file_df)
n_clusters = max(2, min(5, n_samples - 1)) if n_samples > 2 else 2
pheno.compute_clustering(n_clusters=n_clusters, clustering_method="kmeans")
pheno.plot_pca(color_by="cluster")

enrichment_df = pheno.analyze_cluster_enrichment(cluster_col="cluster")
if enrichment_df.empty:
    print("Enrichment table empty (need properties for z-scores).")
else:
    for cluster in sorted(enrichment_df["Cluster"].unique())[:5]:
        sub = enrichment_df[enrichment_df["Cluster"] == cluster].head(5)
        print(f"\nCluster {cluster} (top properties by |z-score|):")
        print(sub[["Property", "Score"]].to_string(index=False))

## 5. Representative (prototype) images

Prototypes are the images **closest to each group’s centroid** in embedding space—useful to show a “typical” field or cell for each cluster.

In [ ]:
prototypes = pheno.find_prototypes(cluster_col="cluster", n_prototypes=3)
for name, idxs in prototypes.items():
    if not idxs:
        continue
    print(f"Cluster {name}: prototype index {idxs[0]}")
    pheno.plot_image_by_index(idxs[0], show_extra_info=True)
    break  # remove break to plot one prototype per cluster

## 6. Distance to reference and outliers

**Distance:** how far each image (or group) lies from your **reference** centroid in embedding space. **Outliers:** images unusually far from their group’s centroid—good candidates for visual QC.

In [ ]:
distance_results = pheno.compute_reference_distances(
    reference_filters=reference_filters,
    source="embeddings",
    mode="centroid",
)
pheno.plot_distance_distribution(distance_results, group_by="condition")

outliers = pheno.detect_outliers(group_by="condition", threshold=3.0)
if outliers["outlier_indices"]:
    top_idx = outliers["outlier_indices"][0]
    print(f"Example outlier index: {top_idx}")
    pheno.plot_image_by_index(top_idx, show_extra_info=True)
else:
    print("No outliers detected with current settings.")

## 7. Two kinds of explainability (needs properties)

- **`compute_component_correlation`:** relates **PCA / t-SNE / UMAP axes** to your measured properties—answers *which biology aligns with this plot axis?*
- **`compute_embedding_property_correlations`** (plus `aggregate_embedding_property_correlations` and `plot_property_correlations`): summarizes how each property co-varies with **embedding dimensions**—a complementary, global view.

The interactive beginner tutorial also runs component–property correlations with widgets ([01_beginner_interactive.ipynb](../tutorials/01_beginner_interactive.ipynb)).

In [ ]:
subset_keys = pheno.get_available_property_keys()[:20]
if not subset_keys:
    print("Skip: need compute_properties first.")
else:
    corr_comp = pheno.compute_component_correlation(
        method="pca",
        n_components=3,
        top_k=10,
    )
    if corr_comp and "summary" in corr_comp and not corr_comp["summary"].empty:
        print(corr_comp["summary"].head(10).to_string(index=False))
    else:
        print("No component correlation summary.")

    emb_corr = pheno.compute_embedding_property_correlations(property_keys=subset_keys)
    emb_agg = pheno.aggregate_embedding_property_correlations(emb_corr, top_k=15)
    pheno.plot_property_correlations(emb_agg, top_k=15)

## 8. Resume work and full HTML report

**Checkpoints:** After `save_results`, you can run `find_files` again (same discovery order as before) and then `pheno.load_results("results/phenome_results.h5")` to avoid re-embedding. Details: [Pipeline API](../../docs/reference/api/pipeline.md).

**Report:** `generate_report()` produces a standalone HTML file with plots, distances, optional clustering, outliers, and more—see [02_phenotyping_api_usage.ipynb](02_phenotyping_api_usage.ipynb) and the [Report Guide](../../docs/guides/report-guide.md).

## See Also

- [02_phenotyping_api_usage.ipynb](02_phenotyping_api_usage.ipynb): core pipeline + `generate_report`
- [03_cell_painting_preset.ipynb](03_cell_painting_preset.ipynb): Cell Painting–style masks and presets
- [example_advanced_phenotyping.py](example_advanced_phenotyping.py): full script-style workflow (replace all paths with your data)
- [Common Workflows](../../docs/examples/workflows.md)